In [ ]:
"""
Time-lapse nuclear segmentation with manual Napari correction.

Input
- One TIFF hyperstack with axes TZCYX

Output
- *_labels_final.tif       Final segmentation labels with axes TYX
- *_detections_final.csv   Label, centroid, area, and mean intensity per frame

Workflow
1. Select one TZCYX TIFF file.
2. Generate a Lamin Z-maximum projection for every time point.
3. Segment every frame using the fixed settings below.
4. Correct labels manually in Napari.
5. Click "Save finalized labels" to export labels and detections.

Install
    pip install numpy pandas scipy scikit-image tifffile napari magicgui
"""

from pathlib import Path
import re
from tkinter import Tk, filedialog

import napari
import numpy as np
import pandas as pd
import tifffile as tiff
from magicgui import magicgui
from scipy.ndimage import binary_fill_holes, gaussian_filter
from skimage.filters import threshold_li, threshold_sauvola
from skimage.filters.rank import otsu as rank_otsu
from skimage.measure import label as sklabel, regionprops_table
from skimage.morphology import (
    binary_opening,
    disk,
    remove_small_holes,
    remove_small_objects,
)


# =============================================================================
# USER SETTINGS
# =============================================================================

# ---------- Image ----------
LAMIN_CH = 0                    # Lamin channel index in the TZCYX image

# ---------- Segmentation ----------
SEGMENTATION_METHOD = "sauvola" # "li", "sauvola", or "local_otsu"
GAUSSIAN_SIGMA = 1.0            # Gaussian smoothing applied before thresholding
SAUVOLA_WINDOW_SIZE = 31        # Sauvola local window size; must be odd
SAUVOLA_K = 0.0                 # Sauvola sensitivity parameter
LOCAL_OTSU_RADIUS = 15          # Local Otsu neighborhood radius in pixels
MIN_NUCLEAR_AREA_PX = 400       # Remove segmented objects smaller than this area
OPENING_RADIUS_PX = 1           # Morphological opening radius; 0 disables opening
DROP_BORDER_OBJECTS = True      # Remove nuclei touching the image border
BORDER_MARGIN_PX = 0            # Additional border width used for removal


# =============================================================================
# INPUT / OUTPUT
# =============================================================================

def select_tiff_file() -> Path:
    root = Tk()
    root.withdraw()
    selected = filedialog.askopenfilename(
        title="Select TZCYX TIFF file",
        filetypes=[("TIFF files", "*.tif *.tiff")],
    )
    root.destroy()

    if not selected:
        raise SystemExit("No TIFF file selected.")

    return Path(selected)


def save_outputs(
    tif_path: Path,
    labels: np.ndarray,
    detections: pd.DataFrame,
) -> tuple[Path, Path]:
    labels_path = tif_path.parent / f"{tif_path.stem}_labels_final.tif"
    detections_path = tif_path.parent / f"{tif_path.stem}_detections_final.csv"

    tiff.imwrite(
        labels_path,
        labels.astype(np.int16),
        imagej=True,
        metadata={"axes": "TYX"},
    )
    detections.to_csv(detections_path, index=False)

    print(f"Saved: {labels_path}")
    print(f"Saved: {detections_path}")
    return labels_path, detections_path


# =============================================================================
# SEGMENTATION
# =============================================================================

def zmax_channel(data_tzcyx: np.ndarray, channel: int) -> np.ndarray:
    return data_tzcyx[:, :, channel].max(axis=1).astype(np.float32)


def make_odd(value: int) -> int:
    value = max(3, int(value))
    return value if value % 2 else value + 1


def relabel_2d(labels: np.ndarray) -> np.ndarray:
    return sklabel(labels > 0, connectivity=2).astype(np.int32)


def segment_frame(image: np.ndarray) -> np.ndarray:
    smoothed = gaussian_filter(
        image.astype(np.float32),
        GAUSSIAN_SIGMA,
    )

    if SEGMENTATION_METHOD == "li":
        binary = smoothed > threshold_li(smoothed)

    elif SEGMENTATION_METHOD == "sauvola":
        threshold = threshold_sauvola(
            smoothed,
            window_size=make_odd(SAUVOLA_WINDOW_SIZE),
            k=SAUVOLA_K,
        )
        binary = smoothed > threshold

    elif SEGMENTATION_METHOD == "local_otsu":
        low, high = np.percentile(smoothed, (1, 99))

        if high <= low:
            normalized = np.zeros_like(smoothed, dtype=np.float32)
        else:
            normalized = np.clip(
                (smoothed - low) / (high - low),
                0,
                1,
            ).astype(np.float32)

        image_u8 = (normalized * 255).astype(np.uint8)
        threshold = rank_otsu(
            image_u8,
            disk(max(1, int(LOCAL_OTSU_RADIUS))),
        )
        binary = image_u8 > threshold

    else:
        raise ValueError(
            "SEGMENTATION_METHOD must be 'li', 'sauvola', or 'local_otsu'."
        )

    if OPENING_RADIUS_PX > 0:
        binary = binary_opening(
            binary,
            footprint=disk(OPENING_RADIUS_PX),
        )

    binary = binary_fill_holes(binary)
    binary = remove_small_objects(
        binary,
        min_size=MIN_NUCLEAR_AREA_PX,
    )
    binary = remove_small_holes(
        binary,
        area_threshold=max(16, MIN_NUCLEAR_AREA_PX // 5),
    )

    return relabel_2d(binary)


def remove_border_labels(
    labels: np.ndarray,
    margin_px: int,
) -> np.ndarray:
    margin = max(0, int(margin_px))

    border_regions = [
        labels[: margin + 1, :],
        labels[-(margin + 1) :, :],
        labels[:, : margin + 1],
        labels[:, -(margin + 1) :],
    ]
    border_ids = np.unique(
        np.concatenate([region.ravel() for region in border_regions])
    )
    border_ids = border_ids[border_ids != 0]

    output = labels.copy()
    if border_ids.size:
        output[np.isin(output, border_ids)] = 0

    return relabel_2d(output)


def segment_movie(zmax: np.ndarray) -> np.ndarray:
    labels = np.zeros(zmax.shape, dtype=np.int32)

    for frame in range(zmax.shape[0]):
        frame_labels = segment_frame(zmax[frame])

        if DROP_BORDER_OBJECTS:
            frame_labels = remove_border_labels(
                frame_labels,
                BORDER_MARGIN_PX,
            )

        labels[frame] = frame_labels

    return labels


# =============================================================================
# DETECTION MEASUREMENT
# =============================================================================

def measure_frame_regions(
    labels: np.ndarray,
    image: np.ndarray,
    frame: int,
) -> pd.DataFrame:
    properties = regionprops_table(
        labels,
        intensity_image=image,
        properties=[
            "label",
            "area",
            "centroid",
            "mean_intensity",
        ],
    )

    if not len(properties["label"]):
        return pd.DataFrame(
            columns=[
                "frame",
                "x",
                "y",
                "area",
                "mean_intensity",
                "label",
            ]
        )

    table = pd.DataFrame(properties).rename(
        columns={
            "centroid-0": "y",
            "centroid-1": "x",
        }
    )
    table["frame"] = frame

    return table[
        ["frame", "x", "y", "area", "mean_intensity", "label"]
    ]


def measure_movie_regions(
    labels: np.ndarray,
    zmax: np.ndarray,
) -> pd.DataFrame:
    return pd.concat(
        [
            measure_frame_regions(
                labels[frame],
                zmax[frame],
                frame,
            )
            for frame in range(labels.shape[0])
        ],
        ignore_index=True,
    )


# =============================================================================
# MANUAL CORRECTION
# =============================================================================

def open_manual_correction(
    tif_path: Path,
    zmax: np.ndarray,
    initial_labels: np.ndarray,
) -> None:
    viewer = napari.Viewer()
    viewer.dims.ndisplay = 2

    contrast_limits = (
        float(np.percentile(zmax, 2)),
        float(np.percentile(zmax, 98)),
    )

    viewer.add_image(
        zmax,
        name="zmax",
        contrast_limits=contrast_limits,
    )
    labels_layer = viewer.add_labels(
        initial_labels,
        name="seg_labels",
    )

    def current_frame() -> int:
        return int(viewer.dims.current_step[0])

    def remove_ids_from_current_frame(label_ids: list[int]) -> None:
        edited = np.asarray(labels_layer.data).copy()
        frame = current_frame()

        for label_id in label_ids:
            if label_id > 0:
                edited[frame][edited[frame] == label_id] = 0

        labels_layer.data = edited.astype(np.int32)
        labels_layer.refresh()

    @magicgui(
        call_button="Remove Label (current frame)",
        label_id={
            "min": 1,
            "max": 1_000_000,
            "step": 1,
            "value": 1,
        },
    )
    def remove_label(label_id: int):
        remove_ids_from_current_frame([int(label_id)])
        viewer.status = (
            f"Removed label {label_id} from frame {current_frame()}."
        )

    @magicgui(
        call_button="Remove entered label IDs",
        label_ids={
            "label": "Label IDs",
            "value": "",
        },
    )
    def remove_multiple_labels(label_ids: str):
        values = [
            int(value)
            for value in re.findall(r"\d+", label_ids)
        ]

        if not values:
            viewer.status = "Enter one or more label IDs."
            return

        remove_ids_from_current_frame(values)
        viewer.status = (
            f"Removed labels {values} from frame {current_frame()}."
        )

    @magicgui(call_button="Relabel (ALL frames)")
    def relabel_all():
        edited = np.asarray(labels_layer.data).copy()

        for frame in range(edited.shape[0]):
            edited[frame] = relabel_2d(edited[frame])

        labels_layer.data = edited.astype(np.int32)
        labels_layer.refresh()
        viewer.status = "Relabeled all frames."

    @magicgui(call_button="Finalize detections")
    def finalize_detections():
        labels = np.asarray(labels_layer.data).astype(np.int32)
        detections = measure_movie_regions(labels, zmax)

        points = (
            detections[["frame", "y", "x"]].to_numpy(float)
            if not detections.empty
            else np.zeros((0, 3), dtype=float)
        )

        if "detections" in viewer.layers:
            viewer.layers["detections"].data = points
            viewer.layers["detections"].refresh()
        else:
            viewer.add_points(
                points,
                name="detections",
                size=4,
            )

        viewer.status = (
            f"Finalized {len(detections)} detections."
        )

    @magicgui(call_button="Save finalized labels")
    def save_finalized_labels():
        labels = np.asarray(labels_layer.data).astype(np.int32)
        detections = measure_movie_regions(labels, zmax)
        save_outputs(
            tif_path,
            labels,
            detections,
        )
        viewer.status = "Final labels and detections saved."

    viewer.window.add_dock_widget(
        remove_label,
        area="right",
    )
    viewer.window.add_dock_widget(
        remove_multiple_labels,
        area="right",
    )
    viewer.window.add_dock_widget(
        relabel_all,
        area="right",
    )
    viewer.window.add_dock_widget(
        finalize_detections,
        area="right",
    )
    viewer.window.add_dock_widget(
        save_finalized_labels,
        area="right",
    )

    napari.run()


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    tif_path = select_tiff_file()
    data = tiff.imread(tif_path)

    if data.ndim != 5:
        raise ValueError(
            f"Expected TZCYX input, but received shape {data.shape}."
        )

    if not 0 <= LAMIN_CH < data.shape[2]:
        raise ValueError(
            f"LAMIN_CH={LAMIN_CH} is outside the available channel range "
            f"0-{data.shape[2] - 1}."
        )

    zmax = zmax_channel(data, LAMIN_CH)
    labels = segment_movie(zmax)

    open_manual_correction(
        tif_path,
        zmax,
        labels,
    )


if __name__ == "__main__":
    main()

In [ ]:
"""
Time-lapse nuclear tracking and Wrinkling Index analysis.

Input
- Original TIFF hyperstack with axes TZCYX
- *_labels_final.tif generated by Part 1

Output
- *_wrinkle_tracks.csv
- *_edge_maps.tif with axes TCYX:
    channel 0: locally normalized Lamin Z-maximum projection
    channel 1: shrunk nuclear mask
    channel 2: Canny edges before final filtering
    channel 3: wrinkle edges after rim and Feret filtering

Workflow
1. Select one TZCYX TIFF file.
2. Load the finalized labels from Part 1.
3. Run LapTrack automatically using fixed tracking parameters.
4. Compute and display the whole-movie WI preview using fixed parameters.
5. Inspect tracks in Napari.
6. Merge track IDs interactively when necessary.
7. Enter the track IDs to export and click "Export selected tracks".

Install
    pip install numpy pandas scipy scikit-image tifffile napari magicgui laptrack
"""

from pathlib import Path
from typing import Dict, Iterable, Optional, Set, Tuple
import re
from tkinter import Tk, filedialog

import napari
import numpy as np
import pandas as pd
import tifffile as tiff
from laptrack import LapTrack
from magicgui import magicgui
from scipy.ndimage import distance_transform_edt
from skimage import measure
from skimage.feature import canny
from skimage.measure import label as sklabel, regionprops_table
from skimage.morphology import disk, erosion


# =============================================================================
# USER SETTINGS
# =============================================================================

# ---------- Image ----------
LAMIN_CH = 0                    # Lamin channel index in the TZCYX image
DEFAULT_XY_UM = 0.207           # XY pixel size used if TIFF metadata is missing

# ---------- LapTrack ----------
LAP_MAX_DISTANCE_PX = 50.0      # Maximum adjacent-frame linking distance
LAP_GAP_DISTANCE_PX = 30.0      # Maximum gap-closing distance
LAP_SPLIT_DISTANCE_PX = 30.0    # Maximum splitting distance

# ---------- Wrinkling Index ----------
CONTRAST_SATURATION = 0.35      # Percent clipped from each local intensity tail
MASK_SHRINK_FACTOR = 0.90       # Shrink each nuclear mask to 90% scale
CANNY_SIGMA = 1.5               # Gaussian sigma used by the Canny detector
THRESHOLD_MULTIPLIER = 0.5      # k in threshold = (mean - k * SD) / reg
THRESHOLD_REG_FACTOR = 0.75     # Divisor in the adaptive threshold formula
CANNY_HIGH_RATIO = 1.5          # High Canny threshold = low threshold * ratio
RIM_EXCLUDE_PX = 4              # Nuclear rim excluded before final filtering
ELONGATION_MIN = 2.0            # Recorded for compatibility; final filter uses Feret
FERET_MIN_UM = 2.5              # Minimum Feret diameter retained as a wrinkle
CROP_PADDING_PX = 8             # Padding around each nucleus during local WI analysis

# ---------- Interactive defaults ----------
DEFAULT_TRACK_MERGE_TEXT = ""    # Example: "5,12,14\n8,20"
DEFAULT_TRACK_IDS_TO_EXPORT = "all"  # "all", "3,7,12", or ranges such as "5-9"


# =============================================================================
# INPUT / OUTPUT
# =============================================================================

def select_tiff_file() -> Path:
    root = Tk()
    root.withdraw()
    selected = filedialog.askopenfilename(
        title="Select TZCYX TIFF file",
        filetypes=[("TIFF files", "*.tif *.tiff")],
    )
    root.destroy()

    if not selected:
        raise SystemExit("No TIFF file selected.")

    return Path(selected)


def read_xy_pixel_size_um(
    tif_path: Path,
    default_xy: float = DEFAULT_XY_UM,
) -> float:
    with tiff.TiffFile(tif_path) as tif:
        metadata = tif.imagej_metadata or {}
    return float(metadata.get("pixel_width", default_xy))


# =============================================================================
# IMAGE AND DETECTION HELPERS
# =============================================================================

def zmax_channel(
    data_tzcyx: np.ndarray,
    channel: int,
) -> np.ndarray:
    return data_tzcyx[:, :, channel].max(axis=1).astype(np.float32)


def measure_regions(
    labels: np.ndarray,
    image: np.ndarray,
    frame: int,
) -> pd.DataFrame:
    properties = regionprops_table(
        labels,
        intensity_image=image,
        properties=[
            "label",
            "area",
            "centroid",
            "mean_intensity",
        ],
    )

    if not len(properties["label"]):
        return pd.DataFrame(
            columns=[
                "frame",
                "x",
                "y",
                "area",
                "mean_intensity",
                "label",
            ]
        )

    table = pd.DataFrame(properties).rename(
        columns={
            "centroid-0": "y",
            "centroid-1": "x",
        }
    )
    table["frame"] = frame

    return table[
        ["frame", "x", "y", "area", "mean_intensity", "label"]
    ]


def add_or_update_image_layer(
    viewer: napari.Viewer,
    name: str,
    data: np.ndarray,
    *,
    colormap: Optional[str] = None,
    opacity: Optional[float] = None,
    blending: Optional[str] = None,
) -> None:
    if name in viewer.layers:
        layer = viewer.layers[name]
        layer.data = data
    else:
        kwargs = {}
        if colormap is not None:
            kwargs["colormap"] = colormap
        if opacity is not None:
            kwargs["opacity"] = opacity
        layer = viewer.add_image(
            data,
            name=name,
            **kwargs,
        )

    if colormap is not None:
        try:
            layer.colormap = colormap
        except Exception:
            pass

    if opacity is not None:
        layer.opacity = opacity

    if blending is not None:
        layer.blending = blending


# =============================================================================
# TRACKING
# =============================================================================

def run_laptrack(detections: pd.DataFrame) -> pd.DataFrame:
    if detections.empty:
        return pd.DataFrame(
            columns=["track_id", "frame", "y", "x"]
        )

    tracker = LapTrack(
        metric="sqeuclidean",
        splitting_metric="sqeuclidean",
        cutoff=LAP_MAX_DISTANCE_PX**2,
        gap_closing_cutoff=LAP_GAP_DISTANCE_PX**2,
        splitting_cutoff=LAP_SPLIT_DISTANCE_PX**2,
    )

    tracks, _, _ = tracker.predict_dataframe(
        detections.copy(),
        coordinate_cols=["y", "x"],
        frame_col="frame",
        only_coordinate_cols=False,
    )
    tracks = tracks.reset_index()

    return (
        tracks[["track_id", "frame", "y", "x"]]
        .sort_values(["track_id", "frame"])
        .reset_index(drop=True)
    )


def parse_track_selection(
    selection: str,
    available_ids: Iterable[int],
) -> Set[int]:
    available = set(int(value) for value in available_ids)

    if not available:
        return set()

    text = (selection or "").strip().lower()

    if text in {"all", "*"}:
        return available

    selected: Set[int] = set()

    for token in [
        item.strip()
        for item in re.split(r"[,\s]+", text)
        if item.strip()
    ]:
        if re.fullmatch(r"-?\d+", token):
            track_id = int(token)
            if track_id in available:
                selected.add(track_id)
            continue

        match = re.fullmatch(r"(-?\d+)-(-?\d+)", token)
        if match:
            start, end = map(int, match.groups())
            step = 1 if start <= end else -1

            selected.update(
                track_id
                for track_id in range(start, end + step, step)
                if track_id in available
            )

    return selected


def parse_merge_text(text: str) -> Dict[int, list[int]]:
    merge_map = {}

    for line in text.splitlines():
        line = line.strip()

        if not line:
            continue

        track_ids = [
            int(value.strip())
            for value in line.split(",")
            if value.strip()
        ]

        if len(track_ids) >= 2:
            merge_map[track_ids[0]] = track_ids[1:]

    return merge_map


def merge_tracks(
    tracks: pd.DataFrame,
    merge_map: Dict[int, list[int]],
) -> pd.DataFrame:
    merged = tracks.copy()

    for master_id, other_ids in merge_map.items():
        for other_id in other_ids:
            merged.loc[
                merged["track_id"] == other_id,
                "track_id",
            ] = master_id

    return (
        merged.sort_values(["track_id", "frame"])
        .reset_index(drop=True)
    )


def build_tracks_by_frame(
    tracks: pd.DataFrame,
    labels: np.ndarray,
) -> Dict[int, pd.DataFrame]:
    rows = []
    frame_count, height, width = labels.shape

    for _, row in tracks.iterrows():
        frame = int(row["frame"])

        if not 0 <= frame < frame_count:
            continue

        y = int(round(float(row["y"])))
        x = int(round(float(row["x"])))

        if not (
            0 <= y < height
            and 0 <= x < width
        ):
            continue

        label_id = int(labels[frame, y, x])

        if label_id <= 0:
            continue

        rows.append(
            {
                "track_id": int(row["track_id"]),
                "frame": frame,
                "y": float(row["y"]),
                "x": float(row["x"]),
                "label_id": label_id,
            }
        )

    if not rows:
        return {}

    table = pd.DataFrame(rows).drop_duplicates(
        subset=["track_id", "frame"]
    )

    return {
        int(frame): group.reset_index(drop=True)
        for frame, group in table.groupby("frame")
    }


# =============================================================================
# WRINKLING INDEX
# =============================================================================

def normalize_local(
    image: np.ndarray,
    region_mask: np.ndarray,
) -> np.ndarray:
    output = np.zeros_like(image, dtype=np.float32)
    values = image[region_mask]

    if not values.size:
        return output

    low, high = np.percentile(
        values,
        (
            CONTRAST_SATURATION,
            100 - CONTRAST_SATURATION,
        ),
    )

    if high <= low:
        return output

    normalized = np.clip(
        (image.astype(np.float32) - low) / (high - low),
        0,
        1,
    ).astype(np.float32)

    output[region_mask] = normalized[region_mask]
    return output


def shrink_mask(mask: np.ndarray) -> np.ndarray:
    mask = mask.astype(bool)

    if not 0 < MASK_SHRINK_FACTOR < 1:
        return mask

    labels = sklabel(mask, connectivity=2)

    if labels.max() == 0:
        return np.zeros_like(mask, dtype=bool)

    output = np.zeros_like(mask, dtype=bool)

    for label_id in range(1, labels.max() + 1):
        region = labels == label_id
        area = int(region.sum())

        if not area:
            continue

        target_area = max(
            1,
            int(round(area * MASK_SHRINK_FACTOR**2)),
        )

        if target_area >= area:
            output |= region
            continue

        distance = distance_transform_edt(region)
        values = distance[region]
        threshold = np.partition(
            values,
            values.size - target_area,
        )[values.size - target_area]

        output |= distance >= threshold

    return output


def filter_edges_by_feret(
    edges: np.ndarray,
    pixel_size_um: float,
) -> np.ndarray:
    if not edges.any():
        return edges

    labels = measure.label(
        edges,
        connectivity=2,
    )
    output = np.zeros_like(
        edges,
        dtype=bool,
    )

    for region in measure.regionprops(labels):
        feret_um = (
            float(
                max(
                    getattr(region, "feret_diameter_max", 0.0),
                    0.0,
                )
            )
            * pixel_size_um
        )

        if feret_um >= FERET_MIN_UM:
            output[labels == region.label] = True

    return output


def get_label_crop(
    labels: np.ndarray,
    label_id: int,
) -> Optional[Tuple[int, int, int, int]]:
    coordinates = np.where(labels == label_id)

    if not coordinates[0].size:
        return None

    y0 = max(
        0,
        int(coordinates[0].min()) - CROP_PADDING_PX,
    )
    y1 = min(
        labels.shape[0],
        int(coordinates[0].max()) + 1 + CROP_PADDING_PX,
    )
    x0 = max(
        0,
        int(coordinates[1].min()) - CROP_PADDING_PX,
    )
    x1 = min(
        labels.shape[1],
        int(coordinates[1].max()) + 1 + CROP_PADDING_PX,
    )

    return y0, y1, x0, x1


def compute_wi_for_labels(
    image: np.ndarray,
    labels: np.ndarray,
    selected_label_ids: Iterable[int],
    pixel_size_um: float,
) -> Tuple[
    np.ndarray,
    np.ndarray,
    np.ndarray,
    np.ndarray,
    dict,
]:
    normalized_output = np.zeros_like(
        image,
        dtype=np.float32,
    )
    shrunk_output = np.zeros_like(
        labels,
        dtype=bool,
    )
    edges_before_output = np.zeros_like(
        labels,
        dtype=bool,
    )
    edges_after_output = np.zeros_like(
        labels,
        dtype=bool,
    )
    stats_by_label = {}

    selected_ids = sorted(
        set(
            int(value)
            for value in selected_label_ids
            if int(value) > 0
        )
    )

    for label_id in selected_ids:
        crop = get_label_crop(
            labels,
            label_id,
        )

        if crop is None:
            continue

        y0, y1, x0, x1 = crop

        crop_labels = labels[y0:y1, x0:x1]
        crop_image = image[y0:y1, x0:x1].astype(np.float32)
        nucleus_mask = crop_labels == label_id

        if not nucleus_mask.any():
            continue

        shrunk_mask = shrink_mask(nucleus_mask)
        normalized = normalize_local(
            crop_image,
            nucleus_mask,
        )
        inner_mask = erosion(
            shrunk_mask,
            disk(max(0, int(RIM_EXCLUDE_PX))),
        )
        values = normalized[shrunk_mask]

        if values.size:
            mean_value = float(values.mean())
            std_value = float(values.std())
            threshold_raw = (
                mean_value
                - THRESHOLD_MULTIPLIER * std_value
            ) / max(THRESHOLD_REG_FACTOR, 1e-6)
            threshold_used = max(
                float(threshold_raw),
                0.0,
            )
        else:
            mean_value = 0.0
            std_value = 0.0
            threshold_used = 0.0

        nucleus_image = np.zeros_like(
            normalized,
            dtype=np.float32,
        )
        nucleus_image[nucleus_mask] = normalized[nucleus_mask]

        edges = canny(
            nucleus_image,
            sigma=CANNY_SIGMA,
            low_threshold=threshold_used,
            high_threshold=threshold_used * CANNY_HIGH_RATIO,
        )
        edges_before = edges & shrunk_mask
        edges_after = filter_edges_by_feret(
            edges_before & inner_mask,
            pixel_size_um,
        )

        normalized_crop = normalized_output[y0:y1, x0:x1]
        normalized_crop[nucleus_mask] = normalized[nucleus_mask]
        normalized_output[y0:y1, x0:x1] = normalized_crop

        shrunk_output[y0:y1, x0:x1] |= shrunk_mask
        edges_before_output[y0:y1, x0:x1] |= edges_before
        edges_after_output[y0:y1, x0:x1] |= edges_after

        stats_by_label[label_id] = {
            "mean_value": mean_value,
            "std_value": std_value,
            "threshold_used": threshold_used,
        }

    return (
        normalized_output,
        shrunk_output,
        edges_before_output,
        edges_after_output,
        stats_by_label,
    )


def compute_whole_movie_preview(
    zmax: np.ndarray,
    labels: np.ndarray,
    tracks: pd.DataFrame,
    pixel_size_um: float,
) -> np.ndarray:
    tracks_by_frame = build_tracks_by_frame(
        tracks,
        labels,
    )

    frame_count, height, width = labels.shape
    output = np.zeros(
        (frame_count, 4, height, width),
        dtype=np.float32,
    )

    for frame, frame_tracks in tracks_by_frame.items():
        label_ids = frame_tracks["label_id"].unique().tolist()

        (
            normalized,
            shrunk_mask,
            edges_before,
            edges_after,
            _,
        ) = compute_wi_for_labels(
            zmax[frame],
            labels[frame],
            label_ids,
            pixel_size_um,
        )

        output[frame, 0] = normalized
        output[frame, 1] = shrunk_mask.astype(np.float32)
        output[frame, 2] = edges_before.astype(np.float32)
        output[frame, 3] = edges_after.astype(np.float32)

    return output


def export_selected_tracks(
    tif_path: Path,
    output_dir: Path,
    zmax: np.ndarray,
    labels: np.ndarray,
    tracks: pd.DataFrame,
    track_selection: str,
    pixel_size_um: float,
) -> None:
    available_ids = sorted(
        map(
            int,
            tracks["track_id"].unique(),
        )
    )
    selected_ids = parse_track_selection(
        track_selection,
        available_ids,
    )

    if not selected_ids:
        raise ValueError("No matching track IDs were selected.")

    selected_tracks = tracks[
        tracks["track_id"].isin(selected_ids)
    ].copy()

    tracks_by_frame = build_tracks_by_frame(
        selected_tracks,
        labels,
    )

    frame_count, height, width = labels.shape
    output = np.zeros(
        (frame_count, 4, height, width),
        dtype=np.float32,
    )
    records = []
    pixel_area_um2 = pixel_size_um**2

    for frame, frame_tracks in tracks_by_frame.items():
        frame_image = zmax[frame].astype(np.float32)
        frame_labels = labels[frame]
        label_ids = frame_tracks["label_id"].unique().tolist()

        (
            normalized,
            shrunk_mask,
            edges_before,
            edges_after,
            stats_by_label,
        ) = compute_wi_for_labels(
            frame_image,
            frame_labels,
            label_ids,
            pixel_size_um,
        )

        output[frame, 0] = normalized
        output[frame, 1] = shrunk_mask.astype(np.float32)
        output[frame, 2] = edges_before.astype(np.float32)
        output[frame, 3] = edges_after.astype(np.float32)

        for _, row in frame_tracks.iterrows():
            label_id = int(row["label_id"])
            nucleus_mask = frame_labels == label_id

            area_before_px = int(nucleus_mask.sum())
            area_after_px = int(
                (shrunk_mask & nucleus_mask).sum()
            )

            if area_after_px:
                edge_count_before = int(
                    (edges_before & nucleus_mask).sum()
                )
                edge_count_after = int(
                    (edges_after & nucleus_mask).sum()
                )

                wi_before = (
                    edge_count_before
                    / area_after_px
                    * 100
                )
                wi_after = (
                    edge_count_after
                    / area_after_px
                    * 100
                )
            else:
                wi_before = 0.0
                wi_after = 0.0

            raw_values = frame_image[nucleus_mask]

            if raw_values.size:
                raw_mean = float(raw_values.mean())
                raw_median = float(np.median(raw_values))
                raw_std = float(raw_values.std())
                raw_min = float(raw_values.min())
                raw_max = float(raw_values.max())
                raw_snr = float(
                    raw_mean / (raw_std + 1e-6)
                )
                raw_cv = float(
                    raw_std / (raw_mean + 1e-6)
                )
            else:
                raw_mean = 0.0
                raw_median = 0.0
                raw_std = 0.0
                raw_min = 0.0
                raw_max = 0.0
                raw_snr = 0.0
                raw_cv = 0.0

            stats = stats_by_label.get(
                label_id,
                {
                    "mean_value": 0.0,
                    "std_value": 0.0,
                    "threshold_used": 0.0,
                },
            )

            records.append(
                {
                    "track_id": int(row["track_id"]),
                    "frame": int(frame),
                    "label_id": label_id,
                    "y": float(row["y"]),
                    "x": float(row["x"]),
                    "Area_PX_Before": area_before_px,
                    "Area_PX_After": area_after_px,
                    "Area_um2_Before": (
                        area_before_px * pixel_area_um2
                    ),
                    "Area_um2_After": (
                        area_after_px * pixel_area_um2
                    ),
                    "Mask_Scale": MASK_SHRINK_FACTOR,
                    "Sigma_Canny": CANNY_SIGMA,
                    "Thr_Multiplier": THRESHOLD_MULTIPLIER,
                    "RegFactor": THRESHOLD_REG_FACTOR,
                    "Rim_Exclude_px": RIM_EXCLUDE_PX,
                    "Elongation_Min": ELONGATION_MIN,
                    "Feret_Min_um": FERET_MIN_UM,
                    "PixelSize_um": pixel_size_um,
                    "RawMean": raw_mean,
                    "RawMedian": raw_median,
                    "RawStd": raw_std,
                    "RawMin": raw_min,
                    "RawMax": raw_max,
                    "RawSNR": raw_snr,
                    "RawCV": raw_cv,
                    "Mean_InsideMask": stats["mean_value"],
                    "Std_InsideMask": stats["std_value"],
                    "Threshold_Used": stats["threshold_used"],
                    "WI_Before(%)": wi_before,
                    "WI_After(%)": wi_after,
                }
            )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    csv_path = output_dir / f"{tif_path.stem}_wrinkle_tracks.csv"
    pd.DataFrame(records).sort_values(
        ["track_id", "frame"]
    ).to_csv(
        csv_path,
        index=False,
    )

    edge_map_path = output_dir / f"{tif_path.stem}_edge_maps.tif"
    tiff.imwrite(
        edge_map_path,
        (
            np.clip(output, 0, 1)
            * 255
        ).astype(np.uint8),
        imagej=True,
        metadata={"axes": "TCYX"},
    )

    print(f"Exported track IDs: {sorted(selected_ids)}")
    print(f"Saved: {csv_path}")
    print(f"Saved: {edge_map_path}")


# =============================================================================
# NAPARI TRACK QC
# =============================================================================

def open_track_qc(
    tif_path: Path,
    output_dir: Path,
    zmax: np.ndarray,
    labels: np.ndarray,
    detections: pd.DataFrame,
    tracks: pd.DataFrame,
    pixel_size_um: float,
) -> None:
    viewer = napari.Viewer()
    viewer.dims.ndisplay = 2

    contrast_limits = (
        float(np.percentile(zmax, 2)),
        float(np.percentile(zmax, 98)),
    )

    viewer.add_image(
        np.ascontiguousarray(
            zmax,
            dtype=np.float32,
        ),
        name="zmax",
        contrast_limits=contrast_limits,
    )
    viewer.add_labels(
        np.ascontiguousarray(labels),
        name="seg_labels",
    )
    viewer.add_points(
        detections[
            ["frame", "y", "x"]
        ].to_numpy(float),
        name="detections",
        size=4,
    )

    state = {
        "tracks": tracks.copy(),
    }

    def refresh_tracks_layer() -> None:
        track_array = state["tracks"][
            ["track_id", "frame", "y", "x"]
        ].to_numpy(float)

        if "tracks_laptrack" in viewer.layers:
            viewer.layers["tracks_laptrack"].data = track_array
            viewer.layers["tracks_laptrack"].refresh()
        else:
            viewer.add_tracks(
                track_array,
                name="tracks_laptrack",
            )

    refresh_tracks_layer()

    whole_movie_preview = compute_whole_movie_preview(
        zmax,
        labels,
        state["tracks"],
        pixel_size_um,
    )

    add_or_update_image_layer(
        viewer,
        "WI zmax_norm_local",
        whole_movie_preview[:, 0],
    )
    add_or_update_image_layer(
        viewer,
        "WI mask_shrunk",
        whole_movie_preview[:, 1],
        colormap="cyan",
        opacity=0.25,
    )
    add_or_update_image_layer(
        viewer,
        "WI edges (before)",
        whole_movie_preview[:, 2],
        colormap="yellow",
        blending="additive",
    )
    add_or_update_image_layer(
        viewer,
        "WI edges (after)",
        whole_movie_preview[:, 3],
        colormap="green",
        blending="additive",
    )

    @magicgui(
        call_button="Apply track merge",
        merge_text={
            "widget_type": "TextEdit",
            "label": "Merge tracks: master,id1,id2",
            "value": DEFAULT_TRACK_MERGE_TEXT,
        },
    )
    def merge_tracks_panel(merge_text: str):
        merge_map = parse_merge_text(merge_text)

        if not merge_map:
            viewer.status = "No valid merge instructions."
            return

        state["tracks"] = merge_tracks(
            state["tracks"],
            merge_map,
        )
        refresh_tracks_layer()
        viewer.status = f"Applied merges: {merge_map}"

    @magicgui(
        call_button="Export selected tracks",
        track_ids={
            "label": "Track IDs: all | 3,7,12 | 5-9",
            "value": DEFAULT_TRACK_IDS_TO_EXPORT,
        },
    )
    def export_tracks_panel(track_ids: str):
        export_selected_tracks(
            tif_path,
            output_dir,
            zmax,
            labels,
            state["tracks"],
            track_ids,
            pixel_size_um,
        )
        viewer.status = "Selected tracks exported."

    viewer.window.add_dock_widget(
        merge_tracks_panel,
        area="right",
    )
    viewer.window.add_dock_widget(
        export_tracks_panel,
        area="right",
    )

    napari.run()


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    tif_path = select_tiff_file()
    labels_path = (
        tif_path.parent
        / f"{tif_path.stem}_labels_final.tif"
    )

    if not labels_path.exists():
        raise FileNotFoundError(
            f"Final labels were not found: {labels_path}"
        )

    data = tiff.imread(tif_path)

    if data.ndim != 5:
        raise ValueError(
            f"Expected TZCYX input, but received shape {data.shape}."
        )

    if not 0 <= LAMIN_CH < data.shape[2]:
        raise ValueError(
            f"LAMIN_CH={LAMIN_CH} is outside the available channel range "
            f"0-{data.shape[2] - 1}."
        )

    zmax = zmax_channel(
        data,
        LAMIN_CH,
    )
    labels = tiff.imread(
        labels_path
    ).astype(np.int32)

    if labels.ndim != 3:
        raise ValueError(
            f"Labels must have axes TYX, but received shape {labels.shape}."
        )

    if labels.shape != zmax.shape:
        raise ValueError(
            f"Labels shape {labels.shape} does not match "
            f"Z-maximum projection shape {zmax.shape}."
        )

    detections = pd.concat(
        [
            measure_regions(
                labels[frame],
                zmax[frame],
                frame,
            )
            for frame in range(labels.shape[0])
        ],
        ignore_index=True,
    )

    if detections.empty:
        raise ValueError(
            "No detections were found in the finalized labels."
        )

    tracks = run_laptrack(detections)
    pixel_size_um = read_xy_pixel_size_um(
        tif_path
    )

    print(
        f"Pixel size: {pixel_size_um:.4f} µm"
    )
    print(
        f"LapTrack found {tracks['track_id'].nunique()} tracks."
    )
    print(
        "Whole-movie WI preview is being generated with fixed parameters."
    )

    open_track_qc(
        tif_path,
        tif_path.parent,
        zmax,
        labels,
        detections,
        tracks,
        pixel_size_um,
    )


if __name__ == "__main__":
    main()

In [ ]:
"""
Combine exported time-lapse WI tracks, normalize to a baseline, and create
a publication-ready dual-axis graph.

Input
- One or more folders containing *_wrinkle_tracks.csv files from Part 2

Output
- Combined normalized CSV
- PNG graph showing observed ΔWI, expected ΔWI, and normalized nuclear area
"""

from __future__ import annotations

from pathlib import Path
from typing import List, Optional

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd


# =============================================================================
# USER SETTINGS
# =============================================================================

# ---------- Input datasets ----------
# ---------- Input datasets ----------
DATASETS = [
    {
        "folder": r"F:\path\to\dataset1",  # Folder containing *_wrinkle_tracks.csv files
        "frame_interval_sec": 30,      # Original acquisition interval (seconds per frame)
        "label": "30sec_dataset",      # Dataset name used internally to keep track IDs unique
    },

    # Example of adding another dataset
    # {
    #     "folder": r"F:\path\to\dataset2",   # Folder containing *_wrinkle_tracks.csv files
    #     "frame_interval_sec": 15,                  # Original acquisition interval (seconds per frame)
    #     "label": "15sec_dataset",                  # Unique dataset name
    # },
]

CSV_PATTERN = "*_wrinkle_tracks.csv"      # Filename pattern of Part 2 output CSVs

SAVE_FOLDER = Path(DATASETS[0]["folder"]) # Folder where the combined CSV and plot will be saved
                                          # (Change this if you want to save somewhere else.)

CSV_PATTERN = "*_wrinkle_tracks.csv"       # Part 2 output filename pattern
SAVE_FOLDER = Path(DATASETS[0]["folder"])  # Folder for combined CSV and graph

# ---------- Time alignment and filtering ----------
COMMON_INTERVAL_SEC = 30       # Common interval used to align all datasets
T_START = 0                    # Baseline common frame
T_END = 30                     # Final common frame included
TIME_REQUIRED_SEC = 600        # Track must reach this time after baseline

# ---------- Normalization ----------
METRICS = [
    "WI_After(%)",
    "Area_um2_After",
]                                      # Metrics normalized to T_START
RIGHT_AXIS_METRIC = "Area_um2_After_norm"  # Metric plotted on right Y-axis

# ---------- Axis labels and limits ----------
X_LABEL = "Time (min)"                     # X-axis label
Y1_LABEL = "ΔWI (%)"                      # Left Y-axis label
Y2_LABEL = "Normalized cross-sectional area"  # Right Y-axis label
Y1_LIMITS = (-1, 10)                      # Left Y-axis limits
Y2_LIMITS = (0.2, 1.2)                    # Right Y-axis limits
PLOT_TITLE = None                         # Plot title; None removes it
RELATIVE_TO_TSTART = True                 # Show T_START as time zero
TIME_INTERVAL_MIN = COMMON_INTERVAL_SEC / 60.0  # Minutes per common frame

# ---------- Figure and saving ----------
FIGSIZE = (7, 5)                          # Figure size in inches
DPI_SCREEN = 150                          # Display resolution
DPI_SAVE = 600                            # Saved PNG resolution
SAVE_PNG = True                           # Save the graph as PNG
PNG_PATH = None                           # None creates an automatic filename
SHOW_PLOT = True                          # Display the graph after saving

# ---------- Font sizes ----------
BASE_FONT_SIZE = 10                       # Default font size
AXIS_LABEL_FONTSIZE = 16                  # Axis-label font size
TITLE_FONTSIZE = 12                       # Plot-title font size
XTICK_LABEL_FONTSIZE = 15                 # X tick-label font size
YTICK_LABEL_FONTSIZE = 15                 # Y tick-label font size
LEGEND_FONTSIZE = 16                      # Legend font size
LEGEND_TITLE_FONTSIZE = 0                 # 0 disables separate title sizing
N_FONTSIZE = 15                           # Sample-size annotation font size

# ---------- Lines and axes ----------
LINE_WIDTH = 1.8                          # Mean-line width
AXES_LINEWIDTH = 1.0                      # Axis-spine width
TICK_LENGTH = 4                           # Major tick length
TICK_WIDTH = 1.0                          # Major tick width

# ---------- Grid ----------
SHOW_GRID = True                          # Show major grid lines
GRID_ALPHA = 0.25                         # Grid transparency
GRID_STYLE = "--"                         # Grid line style

# ---------- Legend ----------
LEGEND_OUTSIDE_TOP = True                 # Place legend above the plot
LEGEND_NCOL = 1                           # Number of legend columns
LEGEND_TITLE = None                       # Legend title; None removes it
TOP_MARGIN_FOR_LEGEND = 0.82              # Top margin reserved for legend

# ---------- Tick intervals ----------
X_MAJOR_TICK_MIN = 5.0                    # Major X tick interval in minutes
X_MINOR_TICK_MIN = 1.0                    # Minor X tick interval in minutes
Y1_MAJOR_TICK = 2.0                       # Major left-Y tick interval
Y1_MINOR_TICK = 0.5                       # Minor left-Y tick interval
Y2_MAJOR_TICK = 0.1                       # Major right-Y tick interval
Y2_MINOR_TICK = 0.05                      # Minor right-Y tick interval

# ---------- Sample-size annotation ----------
SHOW_N = True                             # Display number of tracks
N_FORMAT = "n={n_tracks}"                 # Sample-size text
N_LOCATION = (0.98, 0.98)                 # Position in axes coordinates
N_BOX = True                              # Add translucent background


# =============================================================================
# STYLE
# =============================================================================

def set_publication_style() -> None:
    plt.rcParams.update({
        "figure.dpi": DPI_SCREEN,
        "savefig.dpi": DPI_SAVE,
        "figure.figsize": FIGSIZE,
        "font.size": BASE_FONT_SIZE,
        "axes.labelsize": AXIS_LABEL_FONTSIZE,
        "axes.titlesize": TITLE_FONTSIZE,
        "axes.titleweight": "bold",
        "axes.linewidth": AXES_LINEWIDTH,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": TICK_LENGTH,
        "ytick.major.size": TICK_LENGTH,
        "xtick.major.width": TICK_WIDTH,
        "ytick.major.width": TICK_WIDTH,
        "lines.linewidth": LINE_WIDTH,
        "legend.frameon": False,
        "legend.fontsize": LEGEND_FONTSIZE,
        "grid.alpha": GRID_ALPHA,
    })


# =============================================================================
# DATA LOADING
# =============================================================================

def load_datasets(datasets: list[dict], pattern: str) -> pd.DataFrame:
    tables = []

    for dataset in datasets:
        folder = Path(dataset["folder"])
        frame_interval_sec = float(dataset["frame_interval_sec"])
        dataset_label = str(dataset["label"])
        files = sorted(folder.glob(pattern))

        if not files:
            raise FileNotFoundError(
                f"No CSV files matching {pattern} were found in {folder}."
            )

        for csv_path in files:
            table = pd.read_csv(csv_path)

            required = {
                "frame",
                "track_id",
                "WI_After(%)",
                "Area_um2_After",
            }
            missing = required.difference(table.columns)
            if missing:
                raise ValueError(
                    f"{csv_path.name} is missing columns: {sorted(missing)}"
                )

            table["dataset"] = dataset_label
            table["source_file"] = csv_path.name
            table["original_frame"] = table["frame"].astype(int)
            table["time_sec"] = (
                table["original_frame"] * frame_interval_sec
            )

            keep = np.isclose(
                table["time_sec"] % COMMON_INTERVAL_SEC,
                0,
                atol=1e-6,
            )
            table = table[keep].copy()
            table["frame"] = (
                table["time_sec"] / COMMON_INTERVAL_SEC
            ).round().astype(int)

            table["track_key"] = (
                table["dataset"].astype(str)
                + "||"
                + table["source_file"].astype(str)
                + "||"
                + table["track_id"].astype(str)
            )
            tables.append(table)

    if not tables:
        raise ValueError("No data were loaded.")

    return pd.concat(tables, ignore_index=True)


# =============================================================================
# FILTERING AND NORMALIZATION
# =============================================================================

def select_tracks_present_after_required_time(
    data: pd.DataFrame,
    t_start: int,
    required_time_sec: float,
    common_interval_sec: float,
    t_end: Optional[int],
) -> pd.Index:
    required_frames = int(
        round(required_time_sec / common_interval_sec)
    )
    required_frame = int(t_start) + required_frames

    tracks_at_start = set(
        data.loc[data["frame"] == int(t_start), "track_key"].unique()
    )

    later_data = data[data["frame"] >= required_frame].copy()
    if t_end is not None:
        later_data = later_data[
            later_data["frame"] <= int(t_end)
        ].copy()

    tracks_after_required = set(
        later_data["track_key"].unique()
    )
    kept_tracks = sorted(
        tracks_at_start.intersection(tracks_after_required)
    )

    print(f"Required time: {required_time_sec} sec")
    print(f"Baseline common frame: {t_start}")
    print(f"Minimum required common frame: {required_frame}")
    print(f"Tracks at baseline: {len(tracks_at_start)}")
    print(
        "Tracks present at or after required time: "
        f"{len(tracks_after_required)}"
    )
    print(f"Tracks kept: {len(kept_tracks)}")

    return pd.Index(kept_tracks)


def compute_expected_wi(
    data: pd.DataFrame,
    t_start: int,
) -> pd.DataFrame:
    baseline = data[data["frame"] == int(t_start)][
        ["track_key", "WI_After(%)", "Area_um2_After"]
    ].rename(
        columns={
            "WI_After(%)": "WI0",
            "Area_um2_After": "A0",
        }
    )

    output = data.merge(
        baseline,
        on="track_key",
        how="left",
    )
    output["WI_expected"] = (
        output["WI0"]
        * output["A0"]
        / output["Area_um2_After"]
    )
    output["WIexpected(%)_norm"] = (
        output["WI_expected"] - output["WI0"]
    )
    return output


def normalize_by_baseline(
    data: pd.DataFrame,
    metrics: List[str],
    t_start: int,
    suffix: str = "_norm",
) -> pd.DataFrame:
    baseline = data[data["frame"] == int(t_start)].copy()

    if baseline.empty:
        raise ValueError("No rows were found at T_START.")

    missing = [
        metric for metric in metrics
        if metric not in data.columns
    ]
    if missing:
        raise ValueError(f"Metric columns are missing: {missing}")

    baseline_columns = baseline[
        ["track_key", *metrics]
    ].rename(
        columns={
            metric: f"{metric}__tstart"
            for metric in metrics
        }
    )

    output = data.merge(
        baseline_columns,
        on="track_key",
        how="left",
    )

    for metric in metrics:
        baseline_column = f"{metric}__tstart"
        normalized_column = f"{metric}{suffix}"

        if metric == "WI_After(%)":
            output[normalized_column] = (
                output[metric] - output[baseline_column]
            )
        else:
            output[normalized_column] = (
                output[metric] / output[baseline_column]
            ).replace([np.inf, -np.inf], np.nan)

    return output.dropna(
        subset=[f"{metric}{suffix}" for metric in metrics]
    )


# =============================================================================
# PLOTTING
# =============================================================================

def plot_mean_sem_by_frame(
    data: pd.DataFrame,
    metric: str,
    axis: plt.Axes,
    label: str,
    color: Optional[str],
) -> np.ndarray:
    grouped = data.groupby("frame")[metric]

    mean = grouped.mean()
    count = grouped.count()
    sem = grouped.std(ddof=1) / np.sqrt(
        count.clip(lower=1)
    )

    frames = mean.index.to_numpy()
    values = mean.to_numpy()
    errors = sem.to_numpy()

    axis.plot(
        frames,
        values,
        label=label,
        color=color,
        linewidth=LINE_WIDTH,
    )
    axis.fill_between(
        frames,
        values - errors,
        values + errors,
        color=color,
        alpha=0.25,
        linewidth=0,
    )
    return frames


def apply_tick_locators(
    axis: plt.Axes,
    x_major: Optional[float],
    x_minor: Optional[float],
    y_major: Optional[float],
    y_minor: Optional[float],
) -> None:
    if x_major is not None:
        axis.xaxis.set_major_locator(
            ticker.MultipleLocator(float(x_major))
        )
    if x_minor is not None:
        axis.xaxis.set_minor_locator(
            ticker.MultipleLocator(float(x_minor))
        )
    if y_major is not None:
        axis.yaxis.set_major_locator(
            ticker.MultipleLocator(float(y_major))
        )
    if y_minor is not None:
        axis.yaxis.set_minor_locator(
            ticker.MultipleLocator(float(y_minor))
        )


def create_dual_axis_plot(
    data: pd.DataFrame,
) -> plt.Figure:
    set_publication_style()
    figure, left_axis = plt.subplots(figsize=FIGSIZE)

    observed_frames = plot_mean_sem_by_frame(
        data,
        "WI_After(%)_norm",
        left_axis,
        "ΔWI observed (%)",
        "C0",
    )
    expected_frames = plot_mean_sem_by_frame(
        data,
        "WIexpected(%)_norm",
        left_axis,
        "ΔWI expected (%)",
        "gray",
    )

    left_axis.set_ylabel(
        Y1_LABEL,
        fontsize=AXIS_LABEL_FONTSIZE,
    )
    left_axis.set_ylim(*Y1_LIMITS)

    if SHOW_GRID:
        left_axis.grid(
            True,
            which="major",
            linestyle=GRID_STYLE,
            alpha=GRID_ALPHA,
        )

    right_axis = left_axis.twinx()
    right_axis.spines["right"].set_visible(True)
    right_axis.spines["right"].set_linewidth(
        AXES_LINEWIDTH
    )

    area_frames = plot_mean_sem_by_frame(
        data,
        RIGHT_AXIS_METRIC,
        right_axis,
        Y2_LABEL,
        "C1",
    )
    right_axis.set_ylabel(
        Y2_LABEL,
        fontsize=AXIS_LABEL_FONTSIZE,
        rotation=270,
        labelpad=25,
    )
    right_axis.set_ylim(*Y2_LIMITS)

    frames = np.array(
        sorted(
            set(observed_frames)
            | set(expected_frames)
            | set(area_frames)
        )
    )

    if RELATIVE_TO_TSTART:
        times = (
            frames - int(T_START)
        ) * float(TIME_INTERVAL_MIN)
    else:
        times = frames * float(TIME_INTERVAL_MIN)

    left_axis.set_xticks(frames)
    left_axis.set_xticklabels(
        [
            f"{int(value)}"
            if np.isclose(value, round(value))
            else f"{value:g}"
            for value in times
        ],
        fontsize=XTICK_LABEL_FONTSIZE,
    )
    left_axis.set_xlabel(
        X_LABEL,
        fontsize=AXIS_LABEL_FONTSIZE,
    )

    x_major_frames = (
        None
        if X_MAJOR_TICK_MIN is None
        else X_MAJOR_TICK_MIN / TIME_INTERVAL_MIN
    )
    x_minor_frames = (
        None
        if X_MINOR_TICK_MIN is None
        else X_MINOR_TICK_MIN / TIME_INTERVAL_MIN
    )

    apply_tick_locators(
        left_axis,
        x_major_frames,
        x_minor_frames,
        Y1_MAJOR_TICK,
        Y1_MINOR_TICK,
    )
    apply_tick_locators(
        right_axis,
        None,
        None,
        Y2_MAJOR_TICK,
        Y2_MINOR_TICK,
    )

    left_axis.tick_params(
        axis="y",
        labelsize=YTICK_LABEL_FONTSIZE,
    )
    right_axis.tick_params(
        axis="y",
        labelsize=YTICK_LABEL_FONTSIZE,
    )

    left_lines, left_labels = (
        left_axis.get_legend_handles_labels()
    )
    right_lines, right_labels = (
        right_axis.get_legend_handles_labels()
    )

    legend_title = (
        LEGEND_TITLE
        if LEGEND_TITLE not in ("", None)
        else None
    )

    if LEGEND_OUTSIDE_TOP:
        figure.subplots_adjust(top=TOP_MARGIN_FOR_LEGEND)
        legend = left_axis.legend(
            left_lines + right_lines,
            left_labels + right_labels,
            loc="lower center",
            bbox_to_anchor=(0.5, 1.02),
            ncol=LEGEND_NCOL,
            fontsize=LEGEND_FONTSIZE,
            title=legend_title,
            frameon=False,
        )
    else:
        legend = left_axis.legend(
            left_lines + right_lines,
            left_labels + right_labels,
            loc="best",
            fontsize=LEGEND_FONTSIZE,
            title=legend_title,
            frameon=False,
        )

    if (
        legend is not None
        and legend.get_title() is not None
        and LEGEND_TITLE_FONTSIZE
    ):
        legend.get_title().set_fontsize(
            LEGEND_TITLE_FONTSIZE
        )

    if PLOT_TITLE:
        left_axis.set_title(
            PLOT_TITLE,
            fontsize=TITLE_FONTSIZE,
        )

    if SHOW_N:
        annotation_options = {
            "ha": "right",
            "va": "top",
            "fontsize": N_FONTSIZE,
            "transform": left_axis.transAxes,
        }

        if N_BOX:
            annotation_options["bbox"] = {
                "boxstyle": "round,pad=0.25",
                "facecolor": "white",
                "edgecolor": "none",
                "alpha": 0.6,
            }

        left_axis.text(
            N_LOCATION[0],
            N_LOCATION[1],
            N_FORMAT.format(
                n_tracks=int(data["track_key"].nunique())
            ),
            **annotation_options,
        )

    figure.tight_layout()
    return figure


# =============================================================================
# MAIN
# =============================================================================

def main() -> None:
    SAVE_FOLDER.mkdir(parents=True, exist_ok=True)

    all_data = load_datasets(DATASETS, CSV_PATTERN)

    kept_tracks = select_tracks_present_after_required_time(
        all_data,
        t_start=T_START,
        required_time_sec=TIME_REQUIRED_SEC,
        common_interval_sec=COMMON_INTERVAL_SEC,
        t_end=T_END,
    )

    if not len(kept_tracks):
        raise ValueError(
            "No tracks contain both T_START and the required duration."
        )

    selected_data = (
        all_data[
            all_data["track_key"].isin(kept_tracks)
            & all_data["frame"].between(T_START, T_END)
        ]
        .sort_values(["track_key", "frame"])
        .reset_index(drop=True)
    )

    selected_data = compute_expected_wi(
        selected_data,
        T_START,
    )

    normalized_data = normalize_by_baseline(
        selected_data,
        metrics=METRICS,
        t_start=T_START,
    )

    output_csv = SAVE_FOLDER / (
        f"combined_tracks_common_{COMMON_INTERVAL_SEC}s_"
        f"{T_START}_{T_END}_norm.csv"
    )
    normalized_data.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv}")

    figure = create_dual_axis_plot(normalized_data)

    if SAVE_PNG:
        output_png = (
            Path(PNG_PATH)
            if PNG_PATH is not None
            else SAVE_FOLDER
            / (
                f"combined_plot_common_{COMMON_INTERVAL_SEC}s_"
                f"{T_START}-{T_END}.png"
            )
        )
        figure.savefig(
            output_png,
            dpi=DPI_SAVE,
            bbox_inches="tight",
        )
        print(f"Saved: {output_png}")

    if SHOW_PLOT:
        plt.show()

    plt.close(figure)


if __name__ == "__main__":
    main()